In [7]:
# Question 1
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
print(len(files))

72


In [8]:
# Question 2
from gitsource import GithubRepositoryDataReader
from minsearch import Index

# 1. Read lesson markdown files from the repo
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

print("Number of files:", len(files))

# 2. Convert RawRepositoryFile objects into dictionaries
docs = [
    {
        "filename": f.filename,
        "content": f.content
    }
    for f in files
]

print("First document:")
print(docs[0]["filename"])
print(docs[0]["content"][:300])

# 3. Create minsearch index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# 4. Fit/index the documents
index.fit(docs)

# 5. Search
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

# 6. Print search results
for i, r in enumerate(results, start=1):
    print(i, r["filename"])

# 7. Homework answer: first result
print("First result filename:", results[0]["filename"])

Number of files: 72
First document:
01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search
1 01-agentic-rag/lessons/14-agentic-loop.md
2 01-agentic-rag/lessons/15-frameworks.md
3 01-agentic-rag/lessons/13-function-calling.md
4 01-agentic-rag/lessons/11-agents-intro.md
5 01-agentic-rag/lessons/16-other-frameworks.md
First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [9]:
# Question 3
from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader
from minsearch import Index

load_dotenv(override=True)

# ============================================================
# RAG Assistant
# ============================================================

INSTRUCTIONS = """
Your task is to answer questions based only on the provided lesson content.

Use the context to answer accurately.

If the answer is not found in the context, say:
"I don't know."
"""


class LessonRAG:

    def __init__(
        self,
        index,
        llm_client,
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query=query,
            num_results=num_results
        )

    def build_context(self, search_results):

        lines = []

        for doc in search_results:

            lines.append(
                f"FILE: {doc['filename']}"
            )

            lines.append(
                doc["content"]
            )

            lines.append("")

        return "\n".join(lines)

    def build_prompt(
        self,
        query,
        search_results
    ):

        context = self.build_context(
            search_results
        )

        prompt = f"""
            QUESTION:
            {query}

            CONTEXT:
            {context}
            """

        return prompt

    def llm(self, prompt):

        input_messages = [
            {
                "role": "developer",
                "content": INSTRUCTIONS
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):

        search_results = self.search(query)

        prompt = self.build_prompt(
            query,
            search_results
        )

        response = self.llm(prompt)

        answer = response.output_text

        return answer, response


# ============================================================
# Create OpenAI client
# ============================================================

client = OpenAI()


# ============================================================
# Create assistant
# ============================================================

assistant = LessonRAG(
    index=index,
    llm_client=client,
    model="gpt-5.4-mini"
)


# ============================================================
# Homework Query
# ============================================================

query = (
    "How does the agentic loop keep "
    "calling the model until it stops?"
)

answer, response = assistant.rag(query)

print("\n================ ANSWER ================\n")
print(answer)


# ============================================================
# Show retrieved documents
# ============================================================

print("\n============= TOP RESULTS =============\n")

results = assistant.search(query)

for i, r in enumerate(results, start=1):
    print(i, r["filename"])


# ============================================================
# Token Usage
# ============================================================

print("\n============= USAGE =============\n")

print(response.usage)

# Depending on SDK version:

try:
    print(
        "Input Tokens:",
        response.usage.input_tokens
    )
except:
    pass

try:
    print(
        "Prompt Tokens:",
        response.usage.prompt_tokens
    )
except:
    pass

print("\n=================================\n")


================ ANSWER ================

It keeps looping by checking whether the model returned any `function_call` items.

- It calls the model.
- If the response includes a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn means we're done**.

============= TOP RESULTS =============

1 01-agentic-rag/lessons/14-agentic-loop.md
2 01-agentic-rag/lessons/15-frameworks.md
3 01-agentic-rag/lessons/13-function-calling.md
4 01-agentic-rag/lessons/11-agents-intro.md
5 01-agentic-rag/lessons/16-other-frameworks.md

============= USAGE =============

ResponseUsage(input_tokens=7122, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7223)
Input Tokens: 7122




In [10]:
# Question 4
from gitsource import chunk_documents

chunks = chunk_documents(
    docs,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [5]:
# Question 5
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

chunk_assistant = LessonRAG(
    index=chunk_index,
    llm_client=client,
    model="gpt-5.4-mini"
)
query = (
    "How does the agentic loop keep "
    "calling the model until it stops?"
)

answer, response = chunk_assistant.rag(query)

print(response.usage)

ResponseUsage(input_tokens=2305, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2404)


In [6]:
# Question 6
from dotenv import load_dotenv
load_dotenv(override=True)

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


# 1. Load lesson documents
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = [
    {
        "filename": f.filename,
        "content": f.content,
    }
    for f in files
]

print("Documents:", len(documents))


# 2. Chunk documents
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

print("Chunks:", len(chunks))


# 3. Build minsearch index over chunks
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

chunk_index.fit(chunks)


# 4. Create search tool with counter
search_call_count = 0

def search_course_lessons(query: str) -> str:
    """
    Search the LLM Zoomcamp lesson chunks for relevant information.
    Use this tool when you need course lesson context before answering.
    """
    global search_call_count
    search_call_count += 1

    results = chunk_index.search(
        query=query,
        num_results=5,
    )

    lines = []

    for r in results:
        lines.append(f"FILE: {r['filename']}")
        lines.append(f"START: {r.get('start', '')}")
        lines.append(r["content"])
        lines.append("")

    return "\n".join(lines)


# ============================================================
# 5. Register tool
# ============================================================

agent_tools = Tools()
agent_tools.add_tool(search_course_lessons)


# ============================================================
# 6. Agent Instructions
# ============================================================

instructions = """
You're a course teaching assistant.

Answer the student's question using the search tool.

Make multiple searches with different keywords before answering.
"""


# ============================================================
# 7. Chat Interface & Callback
# ============================================================

chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(
    chat_interface
)


# ============================================================
# 8. Create Runner
# ============================================================

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini"
    )
)


# ============================================================
# 9. Run Agent
# ============================================================

question = """
How does the agentic loop work, and how is it different from plain RAG?
"""

result = runner.loop(
    prompt=question,
    callback=callback,
)


# ============================================================
# 10. Final Answer
# ============================================================

print("\n====================")
print("FINAL ANSWER")
print("====================\n")

print(result)


# ============================================================
# 11. Search Count
# ============================================================

print("\n====================")
print("SEARCH TOOL CALLS")
print("====================\n")

print(search_call_count)

Documents: 72
Chunks: 295
-> Response received


-> Response received



FINAL ANSWER

LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\n\nAnswer the student's question using the search tool.\n\nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='\nHow does the agentic loop work, and how is it different from plain RAG?\n', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG difference lesson agentic loop"}', call_id='call_DYt4qNwMrrYv3zz7JSUVUw8x', name='search_course_lessons', type='function_call', id='fc_0d53328e7f956635006a31cca6e8d0819890f17239eb32b0b0', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"RAG agentic loop tool use retrieve generate revise course"}', call_id='call_Nli58mxFbcK422SpolkvgSxb', name='search_course_lessons', type='function_call', id='fc_0d53328e7f956635006a31cca6e8e0819896a7e6be7a5751bb', namespace=None, status='comple